# Module 12: Full System End-to-End

*Estimated time: ~25 minutes*

This is the final module. You have built every piece of TurboQuant from scratch across
12 modules -- scalar quantization, Walsh-Hadamard transforms, random rotations, the beta
distribution, QJL projections, the two-stage split, and Shannon bounds. Now we run the
actual reference implementation end-to-end and compare it to the toy versions you built.

No exercises here. This is a celebration of what you have built.

In [ ]:
import sys; sys.path.insert(0, "..")
sys.path.insert(0, "../reference-code/turboquant-torch")
from src.helpers import section_header, compare_distortions

import time
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from turboquant import (
    TurboQuant,
    TurboQuantKVCache,
    TurboQuantIndex,
    TurboQuantMSE,
)

torch.manual_seed(42)
print("All imports loaded. Let's run the full system.")

---

## 1. The Real MSE Quantizer

First, let's run the reference MSE quantizer (Stage 1 of TurboQuant) on a proper dataset
and compare against the paper's Table 1, the Shannon bound, and the toy implementation
you built in earlier modules.

In [ ]:
section_header(
    "MSE Quantizer: Reference Implementation",
    "5000 vectors, d=256, bits 1-4"
)

n_vectors = 5000
d = 256
torch.manual_seed(42)
vectors = torch.randn(n_vectors, d)
vectors = vectors / vectors.norm(dim=-1, keepdim=True)  # unit vectors

print(f"Created {n_vectors} unit vectors in d={d}")
print(f"Shape: {vectors.shape}")
print(f"Norm check (first 5): {vectors.norm(dim=-1)[:5].tolist()}")

In [ ]:
# Paper Table 1 values (total MSE distortion for unit vectors)
paper_mse = {1: 0.36, 2: 0.117, 3: 0.030, 4: 0.009}

# Shannon bound: D >= 1/4^b
shannon_bound = {b: 1.0 / (4 ** b) for b in [1, 2, 3, 4]}

# Toy implementation values (what you would get from naive Lloyd-Max
# without RHT -- roughly 2-5x worse than the paper)
toy_mse = {1: 0.50, 2: 0.18, 3: 0.05, 4: 0.013}

# Run the reference implementation
ref_mse = {}
for bits in [1, 2, 3, 4]:
    quantizer = TurboQuantMSE(dim=d, bits=bits, seed=42)
    distortion = quantizer.distortion(vectors).mean().item()
    ref_mse[bits] = distortion

# Display comparison table
print(f"{'Bits':<6} {'Shannon Bound':<16} {'Paper Table 1':<16} {'Reference Code':<16} {'Toy (approx)':<16}")
print("-" * 70)
for b in [1, 2, 3, 4]:
    print(
        f"  {b:<4} {shannon_bound[b]:<16.4f} {paper_mse[b]:<16.4f} "
        f"{ref_mse[b]:<16.4f} {toy_mse[b]:<16.4f}"
    )

In [ ]:
# Use the course helper for a formatted comparison
compare_distortions(
    results={f"b={b}": ref_mse[b] for b in [1, 2, 3, 4]},
    paper_values={f"b={b}": paper_mse[b] for b in [1, 2, 3, 4]},
)

print("\nThe reference code should closely match the paper values.")
print("Small differences come from finite sample size (5000 vectors vs. analytical).")

In [ ]:
# Visualization: all four baselines on one chart
bits_arr = np.array([1, 2, 3, 4])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bits_arr, y=[shannon_bound[b] for b in bits_arr],
    mode="lines+markers", name="Shannon Bound (floor)",
    line=dict(color="#2ecc71", width=3, dash="dash"),
    marker=dict(size=10, symbol="diamond"),
))

fig.add_trace(go.Scatter(
    x=bits_arr, y=[paper_mse[b] for b in bits_arr],
    mode="lines+markers", name="Paper Table 1",
    line=dict(color="#9b59b6", width=2, dash="dot"),
    marker=dict(size=9, symbol="square"),
))

fig.add_trace(go.Scatter(
    x=bits_arr, y=[ref_mse[b] for b in bits_arr],
    mode="lines+markers", name="Reference Code",
    line=dict(color="#3498db", width=3),
    marker=dict(size=10, symbol="circle"),
))

fig.add_trace(go.Scatter(
    x=bits_arr, y=[toy_mse[b] for b in bits_arr],
    mode="lines+markers", name="Toy Implementation",
    line=dict(color="#e74c3c", width=2, dash="dashdot"),
    marker=dict(size=8, symbol="x"),
))

fig.update_layout(
    title="MSE Distortion: Four Baselines Compared",
    xaxis_title="Bits per coordinate",
    yaxis_title="Total MSE Distortion (log scale)",
    yaxis_type="log",
    template="plotly_white",
    height=500, width=800,
    xaxis=dict(tickvals=[1, 2, 3, 4]),
    legend=dict(x=0.55, y=0.98),
)
fig.show()

---

## 2. The Real Two-Stage Quantizer

Now the full TurboQuant: (b-1) MSE bits + 1 QJL bit. The key property is **unbiased**
inner product estimation -- the expected value of the estimate equals the true inner
product.

In [ ]:
section_header(
    "Two-Stage Quantizer: Unbiased Inner Products",
    "(b-1) MSE bits + 1 QJL bit"
)

# Use the same vectors
n_test = 500  # use a subset for inner product testing
x = vectors[:n_test]
y = vectors[n_test:2 * n_test]

# True inner products
true_ips = (x * y).sum(dim=-1)  # element-wise IP for paired vectors

print(f"Testing unbiased estimation with {n_test} vector pairs\n")

for bits in [2, 3, 4]:
    tq = TurboQuant(dim=d, bit_width=bits, unbiased=True, seed=42)
    
    # Quantize both sets
    x_out = tq.quantize(x)
    
    # Estimate inner products: full-precision y against quantized x
    estimated_ips = torch.zeros(n_test)
    for i in range(n_test):
        ip_est = tq.compute_inner_product(y[i], x_out)
        estimated_ips[i] = ip_est[i].item()
    
    # Unbiasedness check: mean estimation error should be ~0
    errors = estimated_ips - true_ips
    mean_error = errors.mean().item()
    std_error = errors.std().item()
    rmse = errors.pow(2).mean().sqrt().item()
    
    print(f"  {bits}-bit TurboQuant (unbiased={True}):")
    print(f"    Mean estimation error: {mean_error:+.6f}  (should be ~0)")
    print(f"    Std of error:          {std_error:.6f}")
    print(f"    RMSE:                  {rmse:.6f}")
    print()

In [ ]:
# Visualize: true vs estimated inner products at 3 bits
tq3 = TurboQuant(dim=d, bit_width=3, unbiased=True, seed=42)
x_out_3 = tq3.quantize(x)

est_ips_3 = torch.zeros(n_test)
for i in range(n_test):
    ip_est = tq3.compute_inner_product(y[i], x_out_3)
    est_ips_3[i] = ip_est[i].item()

fig = go.Figure()

fig.add_trace(go.Scattergl(
    x=true_ips.numpy(), y=est_ips_3.numpy(),
    mode="markers",
    marker=dict(size=3, opacity=0.5, color="#3498db"),
    name="Estimated vs True",
))

# Perfect line
ip_range = [true_ips.min().item() - 0.05, true_ips.max().item() + 0.05]
fig.add_trace(go.Scatter(
    x=ip_range, y=ip_range,
    mode="lines", line=dict(color="red", dash="dash", width=2),
    name="Perfect estimation",
))

fig.update_layout(
    title="3-bit TurboQuant: True vs Estimated Inner Products (unbiased)",
    xaxis_title="True inner product",
    yaxis_title="Estimated inner product",
    template="plotly_white",
    height=500, width=600,
)
fig.show()

---

## 3. Compression Ratio Deep Dive

How much memory do we actually save? The simple answer is 32/b (since we replace 32-bit
floats with b-bit codes), but the real answer includes overhead: stored norms, QJL sign
bits, and padding.

In [ ]:
section_header(
    "Compression Ratio Deep Dive",
    "Exact ratios including all overhead"
)

n_example = 10000
bits_list = [1, 2, 3, 4]

original_bytes = n_example * d * 4  # float32
print(f"Original: {n_example} vectors x {d} dims x 4 bytes = {original_bytes / 1024:.1f} KB\n")

comp_data = []
for bits in bits_list:
    tq = TurboQuant(dim=d, bit_width=bits, unbiased=True, seed=42)
    compressed_bytes = tq.memory_bytes(n_example)
    ratio = original_bytes / compressed_bytes
    
    # Breakdown
    code_bits = bits * d * n_example
    norm_bits = 32 * n_example  # float32 norms
    total_bits = code_bits + norm_bits
    overhead_pct = norm_bits / total_bits * 100
    
    comp_data.append({
        "bits": bits, "compressed_bytes": compressed_bytes,
        "ratio": ratio, "overhead_pct": overhead_pct,
    })
    
    print(
        f"  {bits}-bit: {compressed_bytes / 1024:.1f} KB  "
        f"ratio={ratio:.1f}x  "
        f"(norm overhead: {overhead_pct:.1f}%)"
    )

In [ ]:
# Dual-axis chart: compression ratio vs distortion
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=[f"{c['bits']}-bit" for c in comp_data],
        y=[c["ratio"] for c in comp_data],
        name="Compression Ratio",
        marker_color="#3498db",
        opacity=0.8,
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=[f"{b}-bit" for b in bits_list],
        y=[ref_mse[b] for b in bits_list],
        mode="lines+markers",
        name="MSE Distortion",
        line=dict(color="#e74c3c", width=3),
        marker=dict(size=10),
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Compression Ratio vs. Distortion",
    template="plotly_white",
    height=450, width=700,
    bargap=0.3,
)
fig.update_yaxes(title_text="Compression Ratio (x)", secondary_y=False)
fig.update_yaxes(title_text="MSE Distortion", secondary_y=True, type="log")
fig.show()

---

## 4. KV Cache Compression End-to-End

The first major application: compressing the key-value cache in transformer attention.
We simulate a realistic scenario with multiple attention heads and measure how closely
the compressed attention output matches the original.

In [ ]:
section_header(
    "KV Cache Compression",
    "Compressing keys and values for transformer attention"
)

# Simulate a realistic KV cache
batch_size = 1
num_heads = 4
seq_len = 512
head_dim = 128

torch.manual_seed(42)
keys = torch.randn(batch_size, num_heads, seq_len, head_dim)
values = torch.randn(batch_size, num_heads, seq_len, head_dim)
query = torch.randn(batch_size, num_heads, 1, head_dim)  # single query token

print(f"Keys shape:   {keys.shape}")
print(f"Values shape: {values.shape}")
print(f"Query shape:  {query.shape}")

# Standard attention (ground truth)
def standard_attention(q, k, v, scale=None):
    if scale is None:
        scale = k.shape[-1] ** -0.5
    attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
    attn_weights = F.softmax(attn_weights, dim=-1)
    return torch.matmul(attn_weights, v)

attn_original = standard_attention(query, keys, values)
print(f"\nOriginal attention output shape: {attn_original.shape}")

In [ ]:
# Compare compressed attention at different bit widths
print(f"{'Bits':<8} {'Cosine Similarity':<22} {'RMSE':<16} {'Max Error':<16}")
print("-" * 62)

attn_flat_orig = attn_original.flatten()

for bits in [2, 3, 4]:
    cache = TurboQuantKVCache(head_dim=head_dim, bit_width=bits, seed=42)
    compressed = cache.compress(keys, values)
    attn_compressed = cache.attention(query, compressed)
    
    attn_flat_comp = attn_compressed.flatten()
    
    # Cosine similarity between original and compressed attention outputs
    cos_sim = F.cosine_similarity(
        attn_flat_orig.unsqueeze(0),
        attn_flat_comp.unsqueeze(0),
    ).item()
    
    rmse = (attn_flat_orig - attn_flat_comp).pow(2).mean().sqrt().item()
    max_err = (attn_flat_orig - attn_flat_comp).abs().max().item()
    
    print(f"  {bits:<6} {cos_sim:<22.6f} {rmse:<16.6f} {max_err:<16.6f}")

In [ ]:
# Memory savings at different sequence lengths
section_header(
    "Memory Savings",
    "How much memory does KV cache compression save?"
)

seq_lengths = [1_000, 10_000, 100_000]
bits_for_memory = [2, 3, 4]

print(f"{'Seq Length':<14} {'Bits':<8} {'Original (MB)':<16} {'Compressed (MB)':<18} {'Savings':<10}")
print("-" * 66)

memory_data = []
for seq_len_test in seq_lengths:
    for bits in bits_for_memory:
        cache = TurboQuantKVCache(head_dim=head_dim, bit_width=bits, seed=42)
        orig_mb, comp_mb, ratio = cache.memory_savings(
            batch_size=1, num_heads=32, seq_len=seq_len_test
        )
        savings_pct = (1 - comp_mb / orig_mb) * 100
        memory_data.append({
            "seq_len": seq_len_test, "bits": bits,
            "orig_mb": orig_mb, "comp_mb": comp_mb, "ratio": ratio,
        })
        print(
            f"  {seq_len_test:<12,} {bits:<8} {orig_mb:<16.1f} {comp_mb:<18.1f} "
            f"{savings_pct:.0f}%"
        )

In [ ]:
# Memory savings chart
fig = go.Figure()

colors = {2: "#e74c3c", 3: "#3498db", 4: "#2ecc71"}
for bits in bits_for_memory:
    subset = [m for m in memory_data if m["bits"] == bits]
    fig.add_trace(go.Bar(
        x=[f"{m['seq_len']:,} tokens" for m in subset],
        y=[m["comp_mb"] for m in subset],
        name=f"{bits}-bit compressed",
        marker_color=colors[bits],
    ))

# Add original as reference
orig_mbs = [m["orig_mb"] for m in memory_data if m["bits"] == 3]  # same for all bits
fig.add_trace(go.Bar(
    x=[f"{s:,} tokens" for s in seq_lengths],
    y=orig_mbs,
    name="Original (float32)",
    marker_color="#95a5a6",
    opacity=0.5,
))

fig.update_layout(
    title="KV Cache Memory: Original vs Compressed (32 heads, head_dim=128)",
    xaxis_title="Sequence Length",
    yaxis_title="Memory (MB)",
    yaxis_type="log",
    barmode="group",
    template="plotly_white",
    height=500, width=800,
)
fig.show()

---

## 5. Vector Search End-to-End

The second major application: approximate nearest neighbor search. TurboQuant requires
**zero indexing time** -- no training, no calibration, no clustering. You just add vectors
and search.

In [ ]:
section_header(
    "Vector Search: TurboQuantIndex",
    "Approximate nearest neighbor with zero training"
)

# Database and queries
n_db = 10_000
n_queries = 100
search_dim = 256

torch.manual_seed(42)
database = torch.randn(n_db, search_dim)
database = database / database.norm(dim=-1, keepdim=True)
queries = torch.randn(n_queries, search_dim)
queries = queries / queries.norm(dim=-1, keepdim=True)

# Brute-force ground truth
t0 = time.perf_counter()
true_scores = queries @ database.T
true_top100 = torch.topk(true_scores, 100, dim=-1)
brute_force_time = time.perf_counter() - t0

print(f"Database: {n_db} vectors, dim={search_dim}")
print(f"Queries:  {n_queries}")
print(f"Brute-force time: {brute_force_time * 1000:.1f} ms")

In [ ]:
# Build index and search at different bit widths
def recall_at_k(true_indices, approx_indices, k):
    """Compute recall@k averaged over queries."""
    recalls = []
    for i in range(true_indices.shape[0]):
        true_set = set(true_indices[i, :k].tolist())
        approx_set = set(approx_indices[i, :k].tolist())
        recalls.append(len(true_set & approx_set) / k)
    return np.mean(recalls)

search_results = []

for bits in [2, 3, 4]:
    index = TurboQuantIndex(dim=search_dim, bit_width=bits, seed=42)
    
    # Index -- should be near-instant
    t0 = time.perf_counter()
    index.add(database)
    index_time = time.perf_counter() - t0
    
    # Search
    t0 = time.perf_counter()
    scores, indices = index.search(queries, k=100)
    search_time = time.perf_counter() - t0
    
    # Recall at various k
    r1 = recall_at_k(true_top100.indices, indices, k=1)
    r10 = recall_at_k(true_top100.indices, indices, k=10)
    r100 = recall_at_k(true_top100.indices, indices, k=100)
    
    search_results.append({
        "bits": bits, "r1": r1, "r10": r10, "r100": r100,
        "index_time": index_time, "search_time": search_time,
    })
    
    print(f"  {bits}-bit: recall@1={r1:.2%}  recall@10={r10:.2%}  recall@100={r100:.2%}")
    print(f"         index: {index_time * 1000:.1f} ms  search: {search_time * 1000:.1f} ms")
    print()

In [ ]:
# Recall chart
fig = go.Figure()

k_labels = ["Recall@1", "Recall@10", "Recall@100"]
for res in search_results:
    fig.add_trace(go.Bar(
        x=k_labels,
        y=[res["r1"], res["r10"], res["r100"]],
        name=f"{res['bits']}-bit",
    ))

fig.update_layout(
    title=f"Vector Search Recall ({n_db:,} vectors, d={search_dim})",
    yaxis_title="Recall",
    yaxis=dict(range=[0, 1.05]),
    barmode="group",
    template="plotly_white",
    height=450, width=700,
)
fig.show()

In [ ]:
# Timing comparison
fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"{r['bits']}-bit" for r in search_results],
    y=[r["index_time"] * 1000 for r in search_results],
    name="Index time",
    marker_color="#3498db",
))

fig.add_trace(go.Bar(
    x=[f"{r['bits']}-bit" for r in search_results],
    y=[r["search_time"] * 1000 for r in search_results],
    name="Search time",
    marker_color="#e74c3c",
))

# Add brute-force reference line
fig.add_hline(
    y=brute_force_time * 1000,
    line_dash="dash", line_color="gray",
    annotation_text=f"Brute force: {brute_force_time * 1000:.1f} ms",
)

fig.update_layout(
    title="Indexing and Search Time",
    yaxis_title="Time (ms)",
    barmode="group",
    template="plotly_white",
    height=400, width=600,
)
fig.show()

print("Note: TurboQuant indexing is near-instant because there is no training.")
print("Search time includes decompression + brute-force over compressed representations.")

---

## 6. Your Toy vs. the Reference -- Side by Side

Let's pick one specific vector and trace it through both implementations. The outputs
will not be identical (different RNG seeds, slightly different engineering choices) but
the **structure** is the same.

In [ ]:
section_header(
    "Side-by-Side: Toy vs Reference",
    "Tracing a single vector through both pipelines"
)

# Pick one vector
torch.manual_seed(42)
test_vec = torch.randn(1, d)
test_vec = test_vec / test_vec.norm(dim=-1, keepdim=True)

print(f"Test vector: shape {test_vec.shape}, norm = {test_vec.norm().item():.4f}")
print(f"First 8 values: {test_vec[0, :8].tolist()}")
print()

# === Reference implementation (3-bit MSE) ===
ref_quantizer = TurboQuantMSE(dim=d, bits=3, seed=0)
ref_output = ref_quantizer.quantize(test_vec)
ref_recon = ref_quantizer.dequantize(ref_output)
ref_distortion = (test_vec - ref_recon).pow(2).sum().item()

print("=== Reference Implementation (3-bit MSE) ===")
print(f"  Step 1: Store norm = {ref_output.norms[0].item():.6f}")
print(f"  Step 2: Normalize to unit vector")
print(f"  Step 3: Randomized Hadamard Transform")
print(f"  Step 4: Lloyd-Max scalar quantization ({2**3} levels per coord)")
print(f"  Codes (first 16): {ref_output.codes[0, :16].tolist()}")
print(f"  Reconstruction (first 8): {ref_recon[0, :8].tolist()}")
print(f"  MSE distortion: {ref_distortion:.6f}")
print()

# === Toy implementation (simulated) ===
# This mimics what you built in Modules 02-06:
# normalize -> RHT -> Lloyd-Max quantize -> dequantize -> inverse RHT -> rescale
from turboquant import RandomizedHadamardTransform, LloydMaxCodebook

toy_seed = 99  # different seed = different rotation
toy_rht = RandomizedHadamardTransform(d, seed=toy_seed)
toy_codebook = LloydMaxCodebook(bits=3, dim=toy_rht.padded_dim)

# Step by step
toy_norm = test_vec.norm(dim=-1)
toy_unit = test_vec / toy_norm.unsqueeze(-1)
toy_rotated = toy_rht.forward(toy_unit)
toy_codes = toy_codebook.quantize(toy_rotated)
toy_dequant = toy_codebook.dequantize(toy_codes)
toy_unrotated = toy_rht.inverse(toy_dequant)
toy_recon = toy_unrotated * toy_norm.unsqueeze(-1)
toy_distortion = (test_vec - toy_recon).pow(2).sum().item()

print("=== Toy Implementation (3-bit MSE, different seed) ===")
print(f"  Step 1: Store norm = {toy_norm[0].item():.6f}")
print(f"  Step 2: Normalize to unit vector")
print(f"  Step 3: Randomized Hadamard Transform (seed={toy_seed})")
print(f"  Step 4: Lloyd-Max scalar quantization ({2**3} levels per coord)")
print(f"  Codes (first 16): {toy_codes[0, :16].tolist()}")
print(f"  Reconstruction (first 8): {toy_recon[0, :8].tolist()}")
print(f"  MSE distortion: {toy_distortion:.6f}")

In [ ]:
# Visual comparison: original vs both reconstructions
fig = go.Figure()

coord_range = list(range(32))  # show first 32 coordinates

fig.add_trace(go.Scatter(
    x=coord_range, y=test_vec[0, :32].tolist(),
    mode="lines+markers", name="Original",
    line=dict(color="black", width=2),
    marker=dict(size=4),
))

fig.add_trace(go.Scatter(
    x=coord_range, y=ref_recon[0, :32].tolist(),
    mode="lines+markers", name="Reference (seed=0)",
    line=dict(color="#3498db", width=2),
    marker=dict(size=4),
))

fig.add_trace(go.Scatter(
    x=coord_range, y=toy_recon[0, :32].tolist(),
    mode="lines+markers", name="Toy (seed=99)",
    line=dict(color="#e74c3c", width=2, dash="dash"),
    marker=dict(size=4),
))

fig.update_layout(
    title="Original vs Reconstructed: First 32 Coordinates",
    xaxis_title="Coordinate index",
    yaxis_title="Value",
    template="plotly_white",
    height=400, width=800,
)
fig.show()

print(f"\nReference distortion: {ref_distortion:.6f}")
print(f"Toy distortion:       {toy_distortion:.6f}")
print(f"\nDifferent seeds produce different rotations, so the codes differ.")
print(f"But the STRUCTURE is identical: normalize -> RHT -> Lloyd-Max -> inverse RHT -> rescale.")
print(f"Both achieve similar distortion because the algorithm is the same.")

---

## 7. What You Have Learned -- The Full Picture

Here is the complete dependency graph of everything you built across 12 modules:

```
Embeddings (you knew this)
     |
Dot products measure similarity (Module 01)
     |
Naive quantization destroys geometry (Module 01)
     |
Lloyd-Max: optimal scalar quantization (Module 02)
     |                                       
BUT: need i.i.d. coordinates               JL Lemma: random projections
     |                                       preserve distances (Module 07)
WHT: fast orthogonal transform (Module 03)       |
     |                                       QJL: 1-bit projections give
RHT: random rotation spreads info (Module 04) unbiased IP estimates (Module 08)
     |                                            |
Beta distribution: known in advance (Module 05)   |
     |                                            |
Stage 1: rotate + Lloyd-Max (Module 06)           |
     |____________________________________________|
                       |
             TurboQuant: (b-1) + 1 bit split (Module 09)
                       |
             Near-optimal (Module 10)
                       |
           KV Cache + Vector Search (Module 11)
                       |
           Full system end-to-end (Module 12) <-- YOU ARE HERE
```

Every single piece in this diagram is something you implemented yourself. The reference
code just packages it up with better engineering: proper error handling, device management,
bit packing, and API design. But the core algorithm is exactly what you built.

In [ ]:
section_header(
    "The Full Picture",
    "12 modules, one algorithm, built from scratch"
)

modules = [
    ("01", "Vectors & Quantization", "Dot products, naive quantization fails"),
    ("02", "Scalar Quantization", "Lloyd-Max: optimal for a known distribution"),
    ("03", "Walsh-Hadamard Transform", "O(d log d) orthogonal transform"),
    ("04", "Random Rotations", "RHT spreads information uniformly"),
    ("05", "Beta Distribution", "Coordinates become Beta-distributed (known!)"),
    ("06", "MSE-Optimal Quantizer", "Stage 1: RHT + Lloyd-Max"),
    ("07", "JL Random Projections", "Random projections preserve distances"),
    ("08", "QJL 1-bit Quantizer", "Sign bits give unbiased IP estimates"),
    ("09", "Two-Stage Split", "(b-1) MSE + 1 QJL = TurboQuant"),
    ("10", "Shannon Bounds", "1.4-2.7x of the information-theoretic limit"),
    ("11", "Applications", "KV cache compression + vector search"),
    ("12", "Full System", "End-to-end with the reference implementation"),
]

for num, title, summary in modules:
    print(f"  Module {num}: {title}")
    print(f"           {summary}")
    print()

---

## 8. Where to Go Next

You now have deep understanding of TurboQuant -- not just how to use it, but *why* it
works and *why* it is hard to do much better. Here is where you can go from here:

### Read the full paper
You have the background now. The paper is [arXiv:2504.19874](https://arxiv.org/abs/2504.19874).
Everything in it should feel familiar -- you built every component.

### Try on real data
- Load a HuggingFace embedding model and quantize its outputs
- Compress the KV cache of an actual language model during inference
- Build a vector search index over your own embeddings

### Explore the reference code
- `benchmarks/` -- performance benchmarks and comparisons
- `examples/` -- worked examples for common use cases
- The implementation details: bit packing, device management, codebook caching

### Related work
- **Product Quantization (PQ)**: splits vectors into subvectors, quantizes each with a learned codebook. Needs training data.
- **RaBitQ**: another data-oblivious quantizer with different trade-offs. Focuses on 1-bit quantization.
- **QUIP#**: quantization for LLM weights (not activations). Uses lattice codebooks.
- **ScaNN / FAISS**: production ANN systems. TurboQuant could be a drop-in compression layer for these.

In [ ]:
section_header(
    "Congratulations!",
    "You built TurboQuant from scratch"
)

print("You started with the question: why does naive quantization break vector search?")
print()
print("You now know the answer at every level:")
print("  - Dot products are sensitive to correlated errors across dimensions")
print("  - Random rotations decorrelate coordinates, enabling scalar quantization")
print("  - Lloyd-Max is optimal for scalar quantization when you know the distribution")
print("  - After rotation, coordinates follow a Beta distribution (or Gaussian for large d)")
print("  - QJL sign bits on the residual make inner product estimates unbiased")
print("  - The (b-1) + 1 bit split gives near-optimal distortion with no training")
print("  - Shannon theory confirms: you cannot do much better than this")
print()
print("That is a lot of understanding packed into 12 modules.")
print()
print("The reference code runs exactly the algorithm you built.")
print("The only difference is engineering polish. The ideas are yours now.")